# [skip-ci]

# ViralScan Basic Usage

This vignette walks through a complete ViralScan run from raw FASTQ to results.

## What we will do

1. Verify the `viralscan` CLI is available.
2. Download a small public FASTQ pair (SRR20710651 — a 10x Genomics v3 PBMC sample).
3. Run ViralScan in NCBI-accession mode using Influenza A (`NC_002021.3`) as the viral reference.
4. Inspect the output tables with `pandas`.
5. Load the AnnData result with `scanpy`.

## Prerequisites

- ViralScan installed (`pip install ViralScan`).
- `kb-python` and `snakemake` available on `PATH` (conda environment recommended).
- SRA Toolkit (`prefetch`, `fasterq-dump`) for FASTQ download, **or** substitute your own FASTQ pair.
- `NCBI_EMAIL` environment variable set, or pass `--ncbi-email` below.

**Estimated runtime:** ~10 min on 6 cores (index build + quantification).

In [ ]:
# Verify viralscan is reachable
!viralscan --help

## Step 1 — Download test FASTQ pair

We use SRR20710651 (a publicly available 10x Genomics v3 PBMC library).
The SRA Toolkit must be installed; see https://github.com/ncbi/sra-tools/wiki.

In [ ]:
import subprocess
from pathlib import Path

SRR = "SRR20710651"
fastq_dir = Path("fastq") / SRR
fastq_dir.mkdir(parents=True, exist_ok=True)

# Download and dump FASTQ (skip if files already exist)
r1 = fastq_dir / f"{SRR}_1.fastq"
r2 = fastq_dir / f"{SRR}_2.fastq"

if not r1.exists() or not r2.exists():
    subprocess.run(["prefetch", SRR, "--output-directory", str(fastq_dir.parent)], check=True)
    subprocess.run(
        ["fasterq-dump", str(fastq_dir.parent / SRR), "--outdir", str(fastq_dir), "--split-files"],
        check=True,
    )

print("FASTQ files:")
for f in sorted(fastq_dir.glob("*.fastq")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}  ({size_mb:.1f} MB)")

## Step 2 — Run ViralScan (NCBI accession mode)

We use Influenza A segment 8 (`NC_002021.3`) as our viral reference.
ViralScan will download the FASTA + GTF from NCBI, build a kallisto index,
and quantify the sample — all in one command.

In [ ]:
import os

output_dir = Path("viralscan_output")
r1_path = fastq_dir / f"{SRR}_1.fastq"
r2_path = fastq_dir / f"{SRR}_2.fastq"

cmd = [
    "viralscan",
    "--ncbi-accession", "NC_002021.3",
    "--output", str(output_dir),
    "--sample1", str(r1_path),
    "--sample2", str(r2_path),
    "--umap",
    "--visual",
    "--cores", "6",
]

# NCBI_EMAIL must be set in environment or replace with --ncbi-email flag
env = os.environ.copy()
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError(f"viralscan exited with code {result.returncode}")
print("\nviralscan completed successfully.")

## Step 3 — Inspect the viral summary table

In [ ]:
import pandas as pd

summary_path = output_dir / "results" / "viral_summary.tsv"
summary = pd.read_csv(summary_path, sep="\t")
print(f"Rows: {len(summary)}  Columns: {list(summary.columns)}")
summary.head(10)

In [ ]:
per_cell_path = output_dir / "results" / "per_cell_viral.tsv"
per_cell = pd.read_csv(per_cell_path, sep="\t")
print(f"Rows: {len(per_cell)}")
per_cell.head(5)

## Step 4 — Load AnnData result with scanpy

In [ ]:
import scanpy as sc

h5ad_path = output_dir / "results" / "adata_multimap.h5ad"
adata = sc.read_h5ad(h5ad_path)
print(adata)
print("\nobs columns:", list(adata.obs.columns))
adata.obs.head(5)

## Step 5 — Check report and plots

In [ ]:
report_path = output_dir / "results" / "report.html"
print(f"HTML report: {report_path.resolve()}")
print(f"Report size: {report_path.stat().st_size / 1024:.1f} KB")

plots_dir = output_dir / "results" / "plots"
plots = sorted(plots_dir.glob("*.png"))
print(f"\nPlots generated ({len(plots)}):")
for p in plots:
    print(f"  {p.name}")

In [ ]:
# Display first histogram if matplotlib is available
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

if plots:
    img = mpimg.imread(plots[0])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(plots[0].stem)
    plt.tight_layout()
    plt.show()
else:
    print("No histogram plots found (visual mode may have been skipped).")

## Summary

In this vignette we:

- Downloaded a public 10x PBMC FASTQ pair (SRR20710651).
- Used ViralScan's NCBI accession mode to automatically fetch the Influenza A
  reference (`NC_002021.3`), build a kallisto index, and quantify viral load.
- Inspected `viral_summary.tsv` and `per_cell_viral.tsv` with pandas.
- Loaded the multimapping-corrected AnnData (`adata_multimap.h5ad`) with scanpy.
- Located the self-contained HTML report and histogram plots.

**Next step:** see `cell_type_enrichment.ipynb` to learn how to stratify
viral prevalence by cell type.